<a href="https://colab.research.google.com/github/liwiaflorkiwicz/EEG_dimensiality_reduction/blob/preprocessing/OPSI_preprocessing1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Obliczeniowe podstawy sztucznej inteligencji - projekt

In [33]:
#from google.colab import drive
#drive.mount('/content/drive')

In [34]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("houssamboukhecham/eeg-motor-movementimagery-hierarchical")
print("Path to dataset files:", path)

Path to dataset files: /Users/julia/.cache/kagglehub/datasets/houssamboukhecham/eeg-motor-movementimagery-hierarchical/versions/1


In [35]:
import os
import glob
import pandas as pd

print("Zawartość folderu:")
print(os.listdir(path))

Zawartość folderu:
['eeg-mmi-kaggle']


In [36]:
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)

print(f"Found {len(csv_files)} CSV files.")

# sample file
sample_file = csv_files[0]
print(f"\nReading file: {sample_file}\n")

# read as pandas df
df = pd.read_csv(sample_file)

display(df.head())

Found 6583 CSV files.

Reading file: /Users/julia/.cache/kagglehub/datasets/houssamboukhecham/eeg-motor-movementimagery-hierarchical/versions/1/eeg-mmi-kaggle/S008_Task5_1.csv



,Fc5.,Fc3.,Fc1.,Fcz.,Fc2.,Fc4.,Fc6.,C5..,C3..,C1..,...,P8..,Po7.,Po3.,Poz.,Po4.,Po8.,O1..,Oz..,O2..,Iz..
0,23.0,15.0,5.0,-4.0,-5.0,-5.0,31.0,6.0,-18.0,17.0,...,5.0,15.0,13.0,14.0,31.0,9.0,33.0,-12.0,-2.0,-18.0
1,40.0,23.0,8.0,-2.0,-2.0,-1.0,52.0,11.0,-20.0,15.0,...,14.0,11.0,11.0,15.0,34.0,18.0,33.0,-8.0,8.0,-7.0
2,49.0,33.0,16.0,4.0,4.0,5.0,45.0,30.0,-13.0,21.0,...,12.0,-1.0,1.0,15.0,31.0,15.0,24.0,-13.0,4.0,-10.0
3,58.0,31.0,14.0,3.0,2.0,2.0,43.0,15.0,-18.0,17.0,...,11.0,8.0,5.0,20.0,36.0,14.0,29.0,-10.0,5.0,-6.0
4,53.0,30.0,18.0,7.0,6.0,7.0,37.0,21.0,-13.0,25.0,...,26.0,41.0,27.0,33.0,52.0,27.0,48.0,6.0,22.0,14.0


In [37]:
# !pip install mne

In [38]:
import mne

# sampling frequency
sfreq = 160

# channels names
ch_names = list(df.columns)

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

# data in MNE format (n_channels, n_samples).
data = df.values.T

# raw eeg instance
raw = mne.io.RawArray(data, info)

Creating RawArray with float64 data, n_channels=64, n_times=640
    Range : 0 ... 639 =      0.000 ...     3.994 secs
Ready.


In [39]:
import glob
from collections import Counter

# Listy do przechowywania wyciągniętych informacji
lista_pacjentow = []
lista_zadan = []

for plik in csv_files:
    # 1. Pobieramy samą nazwę pliku (np. S035_Task1_6.csv)
    nazwa_pliku = os.path.basename(plik)

    # 2. Usuwamy rozszerzenie .csv (zostaje: S035_Task1_6)
    nazwa_bez_rozszerzenia = nazwa_pliku.replace('.csv', '')

    # 3. Rozdzielamy nazwę po znaku podkreślenia
    czesci = nazwa_bez_rozszerzenia.split('_')

    # Zabezpieczenie: sprawdzamy, czy nazwa ma dokładnie 3 części
    if len(czesci) >= 2:
        pacjent = czesci[0]  # np. 'S035'
        zadanie = czesci[1]  # np. 'Task1'

        lista_pacjentow.append(pacjent)
        lista_zadan.append(zadanie)

# 4. Zliczanie wystąpień klas
zliczenia_zadan = Counter(lista_zadan)
unikalni_pacjenci = set(lista_pacjentow)

# 5. Wyświetlanie raportu
print("--- RAPORT KLAS W DATASECIE ---")
print(f"Liczba plików: {len(csv_files)}")
print(f"Liczba pacjentów: {len(unikalni_pacjenci)}\n")

print("Klasy (zadania):")
for zadanie, liczba in sorted(zliczenia_zadan.items()):
    print(f" - {zadanie}: {liczba} files")

--- RAPORT KLAS W DATASECIE ---
Liczba plików: 6583
Liczba pacjentów: 109

Klasy (zadania):
 - Task1: 822 files
 - Task2: 825 files
 - Task3: 810 files
 - Task4: 819 files
 - Task5: 837 files
 - Task6: 830 files
 - Task7: 825 files
 - Task8: 815 files


### Sanity check - weryfikacja poprawności danych

- liczba kanałów (64 elektrody EEG)
- częstotliwość próbkowania (160 Hz)

In [40]:
test_file = csv_files[289]
df = pd.read_csv(test_file)
df = pd.read_csv(test_file)

print(f"--- RAPORT POPRAWNOŚCI DANYCH ---")
print(f"Badany plik: {os.path.basename(test_file)}\n")


print("1. LICZBA KANAŁÓW:")
liczba_kolumn = len(df.columns)
print(f"- Całkowita liczba kolumn w pliku: {liczba_kolumn}")
print(f"- Pierwsze 3 kolumny: {list(df.columns)[:3]}")
print(f"- Ostatnie 3 kolumny: {list(df.columns)[-3:]}")


print("\n2. CZĘSTOTLIWOŚĆ PRÓBKOWANIA:")
kolumna_czasu = next((col for col in df.columns if col.lower() in ['time', 'czas']), None)

if kolumna_czasu:
    roznica_czasu = df[kolumna_czasu].iloc[1] - df[kolumna_czasu].iloc[0]
    obliczone_sfreq = round(1 / roznica_czasu)
    print(f"- Odstęp między próbkami wynosi: {roznica_czasu:.5f} sekundy")
    print(f"- Obliczona częstotliwość próbkowania: {obliczone_sfreq} Hz")
    if obliczone_sfreq == 160:
        print("- [OK] Częstotliwość próbkowania zgadza się z oryginalną specyfikacją PhysioNet.")
    else:
        print("- [UWAGA] Częstotliwość jest inna niż 160 Hz.")
else:
    print("Brak kolumny z czasem w pliku. Zakładamy domyślne 160 Hz z opisu datasetu.")
    print(f"- Liczba próbek (wierszy) w pliku: {len(df)}")

--- RAPORT POPRAWNOŚCI DANYCH ---
Badany plik: S073_Task7_8.csv

1. LICZBA KANAŁÓW:
- Całkowita liczba kolumn w pliku: 64
- Pierwsze 3 kolumny: ['Fc5.', 'Fc3.', 'Fc1.']
- Ostatnie 3 kolumny: ['Oz..', 'O2..', 'Iz..']

2. CZĘSTOTLIWOŚĆ PRÓBKOWANIA:
Brak kolumny z czasem w pliku. Zakładamy domyślne 160 Hz z opisu datasetu.
- Liczba próbek (wierszy) w pliku: 640


## Preprocessing



#### Band-pass filter + features extraction

In [41]:
import os
import numpy as np
import pandas as pd
import mne
import scipy.signal
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from mne.decoding import CSP

sfreq = 160
f_low = 8.0
f_high = 45.0

X_features = []
y_labels = []
pominiete_pliki = []

In [42]:
import numpy as np
import scipy.signal
import scipy.stats

def feature_extraction(raw, sfreq):
    data = raw.get_data() # Kształt: (n_channels, n_samples)
    features = []

    freqs, psd = scipy.signal.welch(data, sfreq, nperseg=sfreq)

    bands = {
        'SMR': (8, 13),
        'beta': (13, 20),
        'gamma': (30, 45)}

    for band_name, (b_low, b_high) in bands.items():
        idx_band = np.logical_and(freqs >= b_low, freqs <= b_high)
        features.extend(np.mean(psd[:, idx_band], axis=1)) # Średnia moc w podpaśmie
        features.extend(np.var(psd[:, idx_band], axis=1)) # Wariancja mocy w podpaśmie

    features.extend(scipy.stats.skew(data, axis=1)) # Skośność
    features.extend(scipy.stats.kurtosis(data, axis=1)) # Kurtoza

    return np.array(features)

In [43]:
for idx, file_path in enumerate(csv_files):
    file_name = os.path.basename(file_path).replace('.csv', '')
    parts = file_name.split('_')
    if len(parts) < 2:
        continue
    task_label = parts[1] # nazwa zadania

    try:
        df = pd.read_csv(file_path)

        # Odrzucenie pustych/zbyt krótkich plików
        if df.empty or len(df) < 160:
            pominiete_pliki.append((file_name, "Plik pusty lub za krótki"))
            continue

        ch_names = list(df.columns)
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
        data = df.values.T
        raw = mne.io.RawArray(data, info, verbose=False)

        # Filtrowanie SMR, Beta, Gamma (odcina 60Hz z sieci)
        raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)

        # Ekstrakcja cech pojedynczej epoki
        features = feature_extraction(raw, sfreq)

        X_features.append(features)
        y_labels.append(task_label)

    except Exception as e:
        pominiete_pliki.append((file_name, str(e)))
        continue

    if (idx + 1) % 100 == 0:
        print(f"Przetworzono: {idx + 1}/{len(csv_files)} plików...")

Przetworzono: 100/6583 plików...
Przetworzono: 200/6583 plików...
Przetworzono: 300/6583 plików...
Przetworzono: 400/6583 plików...
Przetworzono: 500/6583 plików...
Przetworzono: 600/6583 plików...
Przetworzono: 700/6583 plików...
Przetworzono: 800/6583 plików...
Przetworzono: 900/6583 plików...
Przetworzono: 1000/6583 plików...
Przetworzono: 1100/6583 plików...
Przetworzono: 1200/6583 plików...
Przetworzono: 1300/6583 plików...


/var/folders/jl/2zwt3w0j4r3bfnp5txqw_j9c0000gn/T/ipykernel_5859/2071634164.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 1400/6583 plików...
Przetworzono: 1500/6583 plików...
Przetworzono: 1600/6583 plików...
Przetworzono: 1700/6583 plików...


/var/folders/jl/2zwt3w0j4r3bfnp5txqw_j9c0000gn/T/ipykernel_5859/2071634164.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 1800/6583 plików...
Przetworzono: 1900/6583 plików...
Przetworzono: 2000/6583 plików...
Przetworzono: 2100/6583 plików...
Przetworzono: 2200/6583 plików...
Przetworzono: 2300/6583 plików...
Przetworzono: 2400/6583 plików...


/var/folders/jl/2zwt3w0j4r3bfnp5txqw_j9c0000gn/T/ipykernel_5859/2071634164.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 2500/6583 plików...


/var/folders/jl/2zwt3w0j4r3bfnp5txqw_j9c0000gn/T/ipykernel_5859/2071634164.py:22: RuntimeWarning: filter_length (265) is longer than the signal (164), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(l_freq=f_low, h_freq=f_high, fir_design='firwin', verbose=False)


Przetworzono: 2600/6583 plików...
Przetworzono: 2700/6583 plików...
Przetworzono: 2800/6583 plików...
Przetworzono: 2900/6583 plików...
Przetworzono: 3000/6583 plików...
Przetworzono: 3100/6583 plików...
Przetworzono: 3200/6583 plików...
Przetworzono: 3300/6583 plików...
Przetworzono: 3400/6583 plików...
Przetworzono: 3500/6583 plików...
Przetworzono: 3600/6583 plików...
Przetworzono: 3700/6583 plików...
Przetworzono: 3800/6583 plików...
Przetworzono: 3900/6583 plików...
Przetworzono: 4000/6583 plików...
Przetworzono: 4100/6583 plików...
Przetworzono: 4200/6583 plików...
Przetworzono: 4300/6583 plików...
Przetworzono: 4400/6583 plików...
Przetworzono: 4500/6583 plików...
Przetworzono: 4600/6583 plików...
Przetworzono: 4700/6583 plików...
Przetworzono: 4800/6583 plików...
Przetworzono: 4900/6583 plików...
Przetworzono: 5000/6583 plików...
Przetworzono: 5100/6583 plików...
Przetworzono: 5200/6583 plików...
Przetworzono: 5300/6583 plików...
Przetworzono: 5400/6583 plików...
Przetworzono: 

In [44]:
# Raport
print("--- RAPORT Z BŁĘDÓW ---")
if pominiete_pliki:
    print(f"Pominięto {len(pominiete_pliki)} uszkodzonych plików.")
else:
    print("Wszystkie pliki zostały przetworzone.")

# Synchronizacja listy csv_files
print(f"Liczba plików PRZED: {len(csv_files)}")
uszkodzone_nazwy = [plik[0] for plik in pominiete_pliki]
csv_files = [f for f in csv_files if os.path.basename(f).replace('.csv', '') not in uszkodzone_nazwy]
print(f"Liczba plików PO: {len(csv_files)}")

--- RAPORT Z BŁĘDÓW ---
Pominięto 40 uszkodzonych plików.
Liczba plików PRZED: 6583
Liczba plików PO: 6543


In [45]:
# Konstrukcja macierzy
X = np.array(X_features)
y = np.array(y_labels)

# Standaryzacja cech
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n--- PREPROCESSING ZAKOŃCZONY ---")
print(f"Kształt macierzy cech X: {X_scaled.shape}")
print(f"Kształt wektora klas y: {y.shape}")


--- PREPROCESSING ZAKOŃCZONY ---
Kształt macierzy cech X: (6543, 512)
Kształt wektora klas y: (6543,)


In [46]:
# Sprawdzenie wartości pustych (NaN) i nieskończonych (Inf)
if np.isnan(X_scaled).any(): print("BŁĄD: Wykryto wartości NaN w X_scaled!")
if np.isinf(X_scaled).any(): print("BŁĄD: Wykryto wartości Inf w X_scaled!")

# Weryfikacja poprawnego działania StandardScaler
if not np.allclose(np.mean(X_scaled, axis=0), 0, atol=1e-7): print("BŁĄD: Średnia cech nie jest równa 0.")
if not np.allclose(np.std(X_scaled, axis=0), 1, atol=1e-7): print("BŁĄD: Odchylenie standardowe nie jest równe 1.")

# Wykrywanie cech o zerowej wariancji (nie niosących żadnej informacji)
zero_var_idx = np.where(np.var(X, axis=0) == 0)[0]
if len(zero_var_idx) > 0: print(f"BŁĄD: Kolumny o zerowej wariancji do usunięcia (indeksy): {zero_var_idx}")

# Sprawdzenie zbalansowania klas dla modelu LDA
_, counts = np.unique(y, return_counts=True)
if len(set(counts)) > 1: print(f"OSTRZEŻENIE: Klasy są niezbalansowane. Rozkład: {dict(zip(*np.unique(y, return_counts=True)))}")

OSTRZEŻENIE: Klasy są niezbalansowane. Rozkład: {np.str_('Task1'): np.int64(818), np.str_('Task2'): np.int64(819), np.str_('Task3'): np.int64(805), np.str_('Task4'): np.int64(814), np.str_('Task5'): np.int64(832), np.str_('Task6'): np.int64(825), np.str_('Task7'): np.int64(819), np.str_('Task8'): np.int64(811)}


In [47]:
import sys

# save preprocessed data to file
liczba_cech = X_scaled.shape[1]
nazwy_kolumn = [f"Cecha_{i+1}" for i in range(liczba_cech)]

df_gotowe = pd.DataFrame(X_scaled, columns=nazwy_kolumn)
df_gotowe['Label'] = y

if 'google.colab' in sys.modules:
    sciezka_csv = '/content/drive/MyDrive/dataset_eeg_features.csv'
else:
    sciezka_csv = 'dataset_eeg_features.csv'

# save to file
df_gotowe.to_csv(sciezka_csv, index=False)

print(f"Dane gotowe na GitHuba zapisane w: {sciezka_csv}")

Dane gotowe na GitHuba zapisane w: dataset_eeg_features.csv


In [48]:
print(df_gotowe.head(20))

     Cecha_1   Cecha_2   Cecha_3   Cecha_4   Cecha_5   Cecha_6   Cecha_7  \
0  -0.578501 -0.574488 -0.595548 -0.596256 -0.564812 -0.394438 -0.386808   
1  -0.592274 -0.542279 -0.421159 -0.451762 -0.376485 -0.330022 -0.435942   
2  -0.792400 -0.825292 -0.820220 -0.793523 -0.793941 -0.566540 -0.807012   
3   0.515849  1.217812  1.431696  1.526452  1.405268  0.582972  0.294542   
4   1.272641  1.923180  2.051120  1.763440  2.013914  0.777735  1.594010   
5  -0.464452 -0.346858 -0.402851 -0.265650 -0.332844 -0.185086 -0.130727   
6   0.069221  0.286154  0.353688  0.203608  0.350876  0.330509  0.424865   
7   0.256505  0.115176 -0.090158 -0.102458 -0.001333  0.026508  0.003957   
8  -0.679742 -0.706308 -0.716461 -0.710734 -0.702043 -0.499620 -0.448506   
9   0.152606  0.292141  0.330772  0.409608  0.425007  0.476563  1.079313   
10 -0.720595 -0.737629 -0.719955 -0.709518 -0.698627 -0.519982 -0.683330   
11 -0.779138 -0.798038 -0.787823 -0.765648 -0.773262 -0.555823 -0.769171   
12 -0.805017